### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [7]:
%pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [8]:
### Open AI API Key and Open Source models--Llama3,Gemma2,mistral--Groq

import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")

groq_api_key=os.getenv("GROQ_API_KEY")
# groq_api_key

In [9]:
%pip install langchain_groq

Note: you may need to restart the kernel to use updated packages.


In [10]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000191A0F9C690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000191A0F9D090>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
%pip install langchain_core

Note: you may need to restart the kernel to use updated packages.


In [12]:
from langchain_core.messages import HumanMessage,SystemMessage

messages=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello How are you?")
]

result=model.invoke(messages)

In [13]:
result

AIMessage(content='Bonjour, comment allez‑vous ?', additional_kwargs={'reasoning_content': 'User wants translation from English to French: "Hello How are you?" Probably "Bonjour, comment ça va?" Or "Bonjour, comment allez-vous?" The user wrote "Hello How are you?" So we translate. Provide French translation. Probably "Bonjour, comment vas-tu ?" or "Bonjour, comment allez-vous ?" The standard polite is "Bonjour, comment allez-vous ?" I\'d answer.'}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 86, 'total_tokens': 180, 'completion_time': 0.096494408, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.004686914, 'prompt_tokens_details': None, 'queue_time': 0.315814059, 'total_time': 0.101181322}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9b8528b477', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0e11b-61d2-7683-b98b-0c3e97906cea-0', tool_calls=[], in

In [14]:
from langchain_core.output_parsers import StrOutputParser
parser=StrOutputParser()
parser.invoke(result)

'Bonjour, comment allez‑vous ?'

In [15]:
### Using LCEL- chain the components
chain=model|parser
chain.invoke(messages)

'Bonjour, comment ça va\u202f?'

In [16]:
### Prompt Templates
from langchain_core.prompts import ChatPromptTemplate

generic_template="Trnaslate the following into {language}:"

prompt=ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{text}")]
)



In [17]:
result=prompt.invoke({"language":"French","text":"Hello"})

In [18]:
result.to_messages()

[SystemMessage(content='Trnaslate the following into French:', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello', additional_kwargs={}, response_metadata={})]

In [19]:
##Chaining together components with LCEL
chain=prompt|model|parser
chain.invoke({"language":"French","text":"Hello"})

'Bonjour'

In [20]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.
